# 📊 Customer Churn Prediction using Machine Learning Pipeline

## 📌 Project Overview

This project implements an **end-to-end machine learning pipeline** to predict customer churn using structured telecom data. The pipeline is built using **Scikit-learn Pipeline API**, ensuring a clean, reusable, and production-ready workflow.

Customer churn prediction is a critical business problem where companies aim to identify customers likely to leave their service.

---

###Install Libraries

In [15]:
# !pip install pandas scikit-learn joblib

Load Dataset

In [16]:
import pandas as pd

# Load dataset
df = pd.read_csv("Telco-Customer-Churn.csv")

# Fix TotalCharges column
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop missing values
df = df.dropna()

# Drop customerID (not useful)
df = df.drop("customerID", axis=1)

# Convert target to binary
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

Separate Features & Target

In [17]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

Identify Column Types

In [18]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

Preprocessing Pipeline

We use:

- Scaling → numeric
- One-hot encoding → categorical

In [19]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

Create Models

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

Full Pipeline

In [21]:
from sklearn.pipeline import Pipeline

pipeline_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipeline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(class_weight="balanced"))
])

Train-Test Split

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

###Hyperparameter Tuning (GridSearch)

Logistic Regression

In [23]:
from sklearn.model_selection import GridSearchCV

param_grid_lr = {
    "model__C": [0.01, 0.1, 1, 10]
}

grid_lr = GridSearchCV(
    pipeline_lr,
    param_grid_lr,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                                       ('model',
                                        LogisticRegression(class_weight='balanced',
                                                           max_iter=1000))]),
             n_jobs=-1, param_grid={'model__C': [0.01, 0.1, 1, 10]},
             scoring='f1')

Random Forest

In [24]:
param_grid_rf = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 20],
    "model__min_samples_split": [2, 5]
}

grid_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                                       ('model',
                                        RandomForestClassifier(class_weight='balanced'))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [5, 10, 20],
                         'model__min_samples_split': [2, 5],
                         'model__n_estimators': [100, 200, 300]},
             scoring='f1')

Evaluation

In [25]:
from sklearn.metrics import accuracy_score, f1_score

# Logistic Regression
y_pred_lr = grid_lr.predict(X_test)
print("LR Accuracy:", accuracy_score(y_test, y_pred_lr))
print("LR F1:", f1_score(y_test, y_pred_lr))

# Random Forest
y_pred_rf = grid_rf.predict(X_test)
print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("RF F1:", f1_score(y_test, y_pred_rf))

LR Accuracy: 0.7313432835820896
LR F1: 0.6078838174273858
RF Accuracy: 0.7569296375266524
RF F1: 0.6068965517241379


In [26]:
#Select Best Model
best_model = grid_rf if grid_rf.best_score_ > grid_lr.best_score_ else grid_lr

In [27]:
import joblib

joblib.dump(best_model, "churn_pipeline.pkl")

['churn_pipeline.pkl']